# Moore vs Mealy machines

Both are FSMs that produce output. In a **Mealy** machine the output depends on the state *and* the current input (it sits on transitions). In a **Moore** machine the output depends on the state only (it sits on states). Here both detect two consecutive `1` bits in a stream, and give identical output streams.

```mermaid
stateDiagram-v2
    direction LR
    state "Mealy" as m
    [*] --> S0
    S0 --> S0: 0 / 0
    S0 --> S1: 1 / 0
    S1 --> S0: 0 / 0
    S1 --> S1: 1 / 1
```

```mermaid
stateDiagram-v2
    direction LR
    [*] --> a
    a --> a: 0
    a --> b: 1
    b --> a: 0
    b --> c: 1
    c --> a: 0
    c --> c: 1
    note right of c: output 1 (a, b output 0)
```

Mealy needs 2 states; Moore needs 3, since it must have a state that *means* "just saw 11".

## Mealy detector
`MealyDetector` is a plain transition table mapping `(state, bit)` to `(next_state, output)`.

In [1]:
from petrilab.fsm.moore_mealy import MealyDetector, MooreDetector

BITS = [0, 1, 1, 0, 1, 1, 1]
mealy_out = MealyDetector().run(BITS)
mealy_out

[0, 0, 1, 0, 0, 1, 1]

The output is 1 whenever the current and previous bits are both 1. Note the output is emitted *on* the transition.

## Moore detector
`MooreDetector` is a `StateChart`; its `output` property is looked up from the active state only.

In [2]:
moore = MooreDetector()
trace = []
for bit in BITS:
    moore.send("one" if bit else "zero")
    trace.append((bit, next(iter(moore.configuration)).id, moore.output))
trace

[(0, 'a', 0),
 (1, 'b', 0),
 (1, 'c', 1),
 (0, 'a', 0),
 (1, 'b', 0),
 (1, 'c', 1),
 (1, 'c', 1)]

Output 1 appears exactly when the machine is in state `c`. Each output is determined by the state alone.

## Equivalence
The two machines should produce the same stream on any input. We check on seeded random streams.

In [3]:
import random

rng = random.Random(0)
ok = True
for _ in range(200):
    bits = [rng.randint(0, 1) for _ in range(rng.randint(0, 30))]
    ok &= MealyDetector().run(bits) == MooreDetector().run(bits)
print("equivalent on 200 random streams:", ok)
print("same on BITS:", mealy_out == MooreDetector().run(BITS))

equivalent on 200 random streams: True
same on BITS: True


Both machines are equivalent as input/output functions, although the Moore version needs an extra state.